# Experimento J01 — decoder OWSM preliminar

Adaptación inicial de OWSM v3.1 a señales neuronales intracorticales. Este cuaderno conserva el intento con decoder de atención y se interpreta como evidencia preliminar, no como una ablación controlada.

No es obtener un buen resultado: es dejar establecido que la adaptación está hecha correctamente y que tiene un fallo estructural

## 0. Instalación

Instalamos espnet 202511 porque es con el que he se han hecho todas las pruebas

In [1]:
!pip install -q "espnet==202511" espnet_model_zoo editdistance h5py

import espnet2, torch
print("ESPnet :", espnet2.__version__)
print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 117.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 29.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 22.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.9 MB/s 

## 1. Configuración


In [ ]:
from google.colab import drive; drive.mount("/content/drive")
import sys
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Carpeta de Drive donde está espnet_eeg.py
CODIGO = "/content/drive/MyDrive/TFG"

if not os.path.exists(f"{CODIGO}/espnet_eeg.py"):
    raise FileNotFoundError(
        f"No encuentro espnet_eeg.py en {CODIGO}\n"
        f"Contenido de esa carpeta: {sorted(os.listdir(CODIGO))}")

if CODIGO not in sys.path:
    sys.path.insert(0, CODIGO)

import espnet_eeg as eeg

import re, glob, json, argparse, logging, string
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import espnetez as ez
from espnet2.bin.s2t_inference import Speech2Text
from espnet2.tasks.s2t import S2TTask

# MODO
CONF = {"n_sesiones": 45, "max_epoch": 20, "warmup": 200, "val_igual_train": False}

DATA_ROOT = "/content/drive/MyDrive/TFG/DatosEEG_crudos/hdf5_data_final"


# Modelo
MODELO   = "espnet/owsm_v3.1_ebf_base"
IDIOMA   = "eng"
FRONTEND = "conv2d" 

# Entrenamiento
BATCH_SIZE  = 16
LR          = 1e-3
GRAD_CLIP   = 5.0
NUM_WORKERS = 2
CTC_WEIGHT  = 1.0          # 1.0 = solo CTC, el decodificador no entra en la pérdida

EXP_DIR   = f"/content/exp/{MODO}"
STATS_DIR = f"/content/exp/stats_{MODO}"
LOG_PATH  = f"{EXP_DIR}/train.log"
os.makedirs(EXP_DIR, exist_ok=True); os.makedirs(STATS_DIR, exist_ok=True)

print(f"Sesiones     : {CONF['n_sesiones']}   ·   val = train: {CONF['val_igual_train']}")
print(f"Épocas       : {CONF['max_epoch']}   ·   warmup: {CONF['warmup']}")
print(f"Frontend     : {FRONTEND}   ·   ctc_weight: {CTC_WEIGHT}")

Mounted at /content/drive
Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.
/usr/local/lib/python3.13/dist-packages/espnet2/enh/encoder/stft_encoder.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/usr/local/lib/python3.13/dist-packages/espnet2/enh/layers/uses2_swin.py:329: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)


MODO         : palabras
Sesiones     : 45   ·   val = train: False
Épocas       : 20   ·   warmup: 200
Frontend     : conv2d   ·   ctc_weight: 1.0


## 2. Los datos

Cada *trial* es un intento de pronunciar una frase. La señal son 512
características por ventana de 20 ms: potencia de banda y tasa de cruces por
umbral de los 256 electrodos.

In [3]:
sesiones = sorted(os.listdir(DATA_ROOT))[:CONF["n_sesiones"]]
print(f"{len(os.listdir(DATA_ROOT))} sesiones disponibles, uso {len(sesiones)}\n")

train_trials = eeg.cargar_split(DATA_ROOT, sesiones, "train")
val_trials   = (train_trials if CONF["val_igual_train"]
                else eeg.cargar_split(DATA_ROOT, sesiones, "val"))

longitudes = np.array([t["input_features"].shape[0] for t in train_trials])
print(f"\nTrain: {len(train_trials)} trials   ·   Val: {len(val_trials)} trials")
print(f"Duración (frames): min {longitudes.min()}  max {longitudes.max()}  "
      f"media {longitudes.mean():.0f}")
print(f"\nEjemplo — shape {train_trials[0]['input_features'].shape}  "
      f"({longitudes[0]*0.02:.1f} s a 50 Hz)")
print(f"          texto '{train_trials[0]['text_raw']}'")

45 sesiones disponibles, uso 45

  t15.2023.08.11/train:  288 trials
  t15.2023.08.13/train:  348 trials
  t15.2023.08.18/train:  197 trials
  t15.2023.08.20/train:  278 trials
  t15.2023.08.25/train:   88 trials
  t15.2023.08.27/train:  150 trials
  t15.2023.09.01/train:  297 trials
  t15.2023.09.03/train:  322 trials
  t15.2023.09.24/train:  245 trials
  t15.2023.09.29/train:  153 trials
  t15.2023.10.01/train:  218 trials
  t15.2023.10.06/train:  174 trials
  t15.2023.10.08/train:  284 trials
  t15.2023.10.13/train:  155 trials
  t15.2023.10.15/train:  239 trials
  t15.2023.10.20/train:   98 trials
  t15.2023.10.22/train:  134 trials
  t15.2023.11.03/train:  149 trials
  t15.2023.11.04/train:   80 trials
  t15.2023.11.17/train:  100 trials
  t15.2023.11.19/train:   60 trials
  t15.2023.11.26/train:  198 trials
  t15.2023.12.03/train:  228 trials
  t15.2023.12.08/train:  198 trials
  t15.2023.12.10/train:  131 trials
  t15.2023.12.17/train:  135 trials
  t15.2023.12.29/train:  198 tr

## 3. Qué espera ESPnet y qué tenemos nosotros

Aquí está el problema de fondo, y conviene verlo antes de tocar nada.

Un modelo de la tarea `s2t` de ESPnet recibe una **forma de onda de una sola
dimensión** a 16 kHz. Su `frontend` la convierte en un banco de filtros mel de
80 dimensiones, y el `encoder.embed` proyecta esas 80 a las 384 del encoder.

Nosotros no tenemos forma de onda. Tenemos una matriz de 512 columnas ya
extraída. Vamos a comprobar exactamente qué dimensiones espera el modelo
haciendo pasar un segundo de audio sintético por su frontend.

In [5]:
pretrained = Speech2Text.from_pretrained(MODELO, lang_sym=f"<{IDIOMA}>", beam_size=5)
pretrain_config = vars(pretrained.s2t_train_args)
tokenizer = pretrained.tokenizer
converter = pretrained.converter
modelo_original = pretrained.s2t_model
modelo_original.eval()

# ── Qué le entra al encoder cuando la entrada es audio ────────────────
with torch.no_grad():
    onda = torch.randn(1, 16000)                       # 1 segundo a 16 kHz
    feats, _ = modelo_original.frontend(onda, torch.tensor([16000]))

print("PIPELINE ORIGINAL DE OWSM")
print("-" * 62)
print(f"  entrada           : forma de onda {tuple(onda.shape)}  (16 kHz, 1-D)")
print(f"  frontend          : {type(modelo_original.frontend).__name__}")
print(f"      -> produce    : {tuple(feats.shape)}   (batch, frames, mel)")
print(f"  specaug           : {type(modelo_original.specaug).__name__}")
print(f"  normalize         : {type(modelo_original.normalize).__name__}")
print(f"  encoder.embed     : {type(modelo_original.encoder.embed).__name__}")
print(f"      -> Linear     : {modelo_original.encoder.embed.out[0]}")
print(f"  encoder d_model   : {modelo_original.encoder.output_size()}")

print()
print("LO QUE TENEMOS NOSOTROS")
print("-" * 62)
print(f"  entrada           : matriz {train_trials[0]['input_features'].shape}  "
      f"(50 Hz, 2-D)")
print(f"  frontend          : ninguno, las características ya están extraídas")
print(f"  dimensión de rasgo: 512  (frente a las {feats.shape[-1]} que espera el embed)")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 29 files:   0%|          | 0/29 [00:00<?, ?it/s]

PIPELINE ORIGINAL DE OWSM
--------------------------------------------------------------
  entrada           : forma de onda (1, 16000)  (16 kHz, 1-D)
  frontend          : DefaultFrontend
      -> produce    : (1, 101, 80)   (batch, frames, mel)
  specaug           : SpecAug
  normalize         : GlobalMVN
  encoder.embed     : Conv2dSubsampling
      -> Linear     : Linear(in_features=7296, out_features=384, bias=True)
  encoder d_model   : 384

LO QUE TENEMOS NOSOTROS
--------------------------------------------------------------
  entrada           : matriz (321, 512)  (50 Hz, 2-D)
  frontend          : ninguno, las características ya están extraídas
  dimensión de rasgo: 512  (frente a las 80 que espera el embed)


El `Conv2dSubsampling` de ESPnet construye su capa de salida como
`Linear(odim · ((idim−1)//2 − 1)//2, odim)`. Con `idim=80` son 7.296 entradas.
Con `idim=512` son 48.768.

Se inicializan a 0

## 4. La adaptación

Cuatro cambios sobre el modelo, todos en `espnet_eeg.adaptar_a_seeg()`:

| # | Componente | ESPnet | Adaptado | Por qué |
|---|---|---|---|---|
| 1 | `model.frontend` | `DefaultFrontend` (STFT → mel) | `None` | Con `frontend=None`, `_extract_feats()` devuelve la entrada tal cual. No hay onda que analizar. |
| 2 | `model.specaug` | `SpecAug` | `None` | Enmascarar «frecuencias» no tiene sentido: el eje son electrodos, no un espectro ordenado. |
| 3 | `model.normalize` | `GlobalMVN` | `None` | Se sustituye por z-score por trial: la ganancia de los electrodos deriva entre sesiones. |
| 4 | `encoder.embed` | `Conv2dSubsampling(80, 384)` | `Conv2dSubsampling(512, 384)` | Única capa cuya entrada depende de la modalidad. |

El resto del modelo —los seis bloques E-Branchformer, la cabeza CTC, el
decodificador— se queda exactamente como está.

> **Detalle que costó tiempo:** si el frontend nuevo no hereda de
> `Conv2dSubsampling`, el encoder lo llama **sin la máscara de relleno**
> (`e_branchformer_encoder.py:483` decide con un `isinstance`) y falla con
> `TypeError: forward() missing 1 required positional argument: 'x_mask'`.
> Está comentado en `espnet_eeg.py`.

In [ ]:
def build_model_fn(args):
    # ── El ctc_weight hay que inyectarlo en model_conf, no sólo arriba ──
    conf = dict(pretrain_config)
    conf["model_conf"] = {**conf.get("model_conf", {}), "ctc_weight": CTC_WEIGHT}

    model = S2TTask.build_model(argparse.Namespace(**conf))
    model.load_state_dict(modelo_original.state_dict(), strict=False)

    # ── LOS CUATRO CAMBIOS ──────────────────────────────────────────────
    eeg.adaptar_a_seeg(model, idim=512, frontend=FRONTEND, verbose=False)

    model.train()
    total, entrenables = eeg.contar_parametros(model)
    print(f"Parámetros totales     : {total:,}")
    print(f"Parámetros entrenables : {entrenables:,} ({entrenables/total*100:.1f} %)")
    return model


print("build_model_fn definido")

build_model_fn definido


Esto es una prueba de que funciona, _m es un modelo que iniciamos y solo le pasamos un trial para ver que funciona

In [7]:
_dev = "cuda" if torch.cuda.is_available() else "cpu"

_conf = dict(pretrain_config)
_conf["model_conf"] = {**_conf.get("model_conf", {}), "ctc_weight": CTC_WEIGHT}
_m = S2TTask.build_model(argparse.Namespace(**_conf))      # todavía es de audio

eeg.adaptar_a_seeg(_m, idim=512, frontend=FRONTEND, verbose=True)

_total, _ = eeg.contar_parametros(_m)
print(f"\nParámetros del modelo adaptado: {_total:,}")

with torch.no_grad():
    _m.eval().to(_dev)
    x = torch.as_tensor(eeg.znorm(train_trials[0]["input_features"])).unsqueeze(0).to(_dev)
    enc, _ = _m.encode(x, torch.tensor([x.size(1)], device=_dev))
    if isinstance(enc, tuple): enc = enc[0]
    logits = _m.ctc.log_softmax(enc)

print(f"\n  señal        {tuple(x.shape)}     (batch, frames, canales)")
print(f"  -> encoder   {tuple(enc.shape)}     (batch, frames/4, d_model)")
print(f"  -> cabeza CTC{tuple(logits.shape)}  (batch, frames/4, vocabulario)")
print("\nEl modelo procesa señal neuronal de extremo a extremo.")

del _m
torch.cuda.empty_cache() if torch.cuda.is_available() else None

componente    ESPnet original           adaptado                  
------------------------------------------------------------------
frontend      DefaultFrontend           NoneType                  
specaug       SpecAug                   NoneType                  
normalize     GlobalMVN                 NoneType                  
embed_in      7296                      48768                     

Parámetros del modelo adaptado: 64,456,018


/usr/local/lib/python3.13/dist-packages/espnet2/s2t/espnet_model.py:338: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(False):



  señal        (1, 321, 512)     (batch, frames, canales)
  -> encoder   (1, 79, 384)     (batch, frames/4, d_model)
  -> cabeza CTC(1, 79, 50002)  (batch, frames/4, vocabulario)

El modelo procesa señal neuronal de extremo a extremo.


## 5. Dataset y `data_info`

In [8]:
train_raw = eeg.SeeGDataset(train_trials, idioma=IDIOMA)
valid_raw = eeg.SeeGDataset(val_trials,   idioma=IDIOMA)

def tokenize(texto):
    return np.array(converter.tokens2ids(tokenizer.text2tokens(texto)))

data_info = {
    "speech":    lambda d: eeg.znorm(d["input_features"]),   # <-- antes: librosa.load(...)
    "text":      lambda d: tokenize(d["text"]),
    "text_prev": lambda d: tokenize(d["text_prev"]),
    "text_ctc":  lambda d: tokenize(d["text_ctc"]),
}

train_dataset = ez.dataset.ESPnetEZDataset(train_raw, data_info=data_info)
valid_dataset = ez.dataset.ESPnetEZDataset(valid_raw, data_info=data_info)

it = train_raw[0]
print("text     :", it["text"])
print("text_ctc :", it["text_ctc"])
print("speech   :", data_info["speech"](it).shape)
print("tokens   :", tokenize(it["text_ctc"]))
print(f"\nTrain: {len(train_dataset)}  ·  Valid: {len(valid_dataset)}")

text     : <eng><asr><notimestamps> bring it closer.
text_ctc : bring it closer
speech   : (321, 512)
tokens   : [ 3083  1697 11977]

Train: 8072  ·  Valid: 1426


## 6. Configuración de entrenamiento

El YAML es el mismo del notebook original, con `specaug: null` (ya no aplica) y
los valores del panel inyectados encima.

In [9]:
yaml_content = f'''
use_lora: false

rir_scp: null
noise_scp: null
speech_volume_normalize: null
non_linguistic_symbols: null

preprocessor_conf:
  speech_name: speech
  text_name: text

seed: 2024
num_workers: {NUM_WORKERS}
ngpu: 1
batch_type: sorted
batch_size: {BATCH_SIZE}
accum_grad: 1
max_epoch: {CONF['max_epoch']}
patience: null
init: null
best_model_criterion:
- [valid, loss, min]
keep_nbest_models: 1
use_amp: true

optim: adam
optim_conf:
    lr: {LR}
    weight_decay: 0.000001
scheduler: warmuplr
scheduler_conf:
    warmup_steps: {CONF['warmup']}

specaug: null
ctc_weight: {CTC_WEIGHT}
grad_clip: {GRAD_CLIP}
'''

with open("finetune_b2t25.yaml", "w") as f:
    f.write(yaml_content)

finetune_config = ez.config.update_finetune_config(
    "s2t", pretrain_config, "finetune_b2t25.yaml")

finetune_config["cudnn_deterministic"] = False    # el True por defecto cuesta ~25 %
finetune_config["multiple_iterator"]   = False
finetune_config["iterator_type"]       = "sequence"
finetune_config["log_interval"]        = 20

finetune_config["batch_type"] = "numel"
finetune_config["batch_bins"] = 4_000_000    # ~9 trials medios, ~3 de los largos
finetune_config["accum_grad"] = 2

NOMBRES = ["speech_shape", "text_shape", "text_prev_shape", "text_ctc_shape"]
finetune_config["train_shape_file"] = [f"{STATS_DIR}/train/{n}" for n in NOMBRES]
finetune_config["valid_shape_file"] = [f"{STATS_DIR}/valid/{n}" for n in NOMBRES]

finetune_config["num_iters_per_epoch"] = None   # una época = una pasada, no lo de OWSM
finetune_config["cudnn_benchmark"] = False      # ver más abajo

pasos = max(1, len(train_dataset) // BATCH_SIZE)
print(f"~{pasos} pasos/época  ·  warmup {CONF['warmup']} pasos "
      f"(~{CONF['warmup']/pasos:.1f} épocas)")
print(f"lr {LR}  ·  batch {BATCH_SIZE}  ·  grad_clip {GRAD_CLIP}  ·  "
      f"ctc_weight {CTC_WEIGHT}")

print("CONFIGURACIÓN EFECTIVA")
for k in ("num_iters_per_epoch", "batch_type", "batch_bins", "accum_grad",
          "max_epoch", "cudnn_benchmark", "cudnn_deterministic"):
    print(f"  {k:22s} {finetune_config.get(k)}")
assert finetune_config["num_iters_per_epoch"] is None, "OWSM está mandando otra vez"

~504 pasos/época  ·  warmup 200 pasos (~0.4 épocas)
lr 0.001  ·  batch 16  ·  grad_clip 5.0  ·  ctc_weight 1.0
CONFIGURACIÓN EFECTIVA
  num_iters_per_epoch    None
  batch_type             numel
  batch_bins             4000000
  accum_grad             2
  max_epoch              20
  cudnn_benchmark        False
  cudnn_deterministic    False


## 7. Entrenamiento

`ez.Trainer` es el de ESPnet-EZ sin modificar. `collect_stats()` recorre el
corpus una vez para escribir los ficheros de formas que necesita el
agrupador por lotes.

In [10]:
trainer = ez.Trainer(
    task="s2t",
    train_config=finetune_config,
    train_dataset=train_dataset,
    valid_dataset=valid_dataset,
    build_model_fn=build_model_fn,
    data_info=data_info,
    output_dir=EXP_DIR,
    stats_dir=STATS_DIR,
    ngpu=1,
)

if all(os.path.exists(p) and os.path.getsize(p) > 0
       for p in finetune_config["train_shape_file"] + finetune_config["valid_shape_file"]):
    print("Los shape files ya existen, salto collect_stats.")
else:
    trainer.collect_stats()

for p in finetune_config["train_shape_file"]:
    print(f"  {os.path.basename(p):18s} {sum(1 for _ in open(p))} líneas")

/usr/bin/python3 /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-865e2ea1-41ad-4bfc-91ee-2b764b05b318.json


Parámetros totales     : 64,456,018
Parámetros entrenables : 64,456,018 (100.0 %)
  speech_shape       8072 líneas
  text_shape         8072 líneas
  text_prev_shape    8072 líneas
  text_ctc_shape     8072 líneas


In [ ]:
# basicConfig es un no-op en Colab (el root ya tiene handlers), hay que forzarlo
raiz = logging.getLogger()
raiz.setLevel(logging.INFO)
raiz.handlers.clear()

fmt = logging.Formatter("%(asctime)s %(message)s", "%H:%M:%S")
for h in (logging.FileHandler(LOG_PATH, mode="w"), logging.StreamHandler(sys.stdout)):
    h.setFormatter(fmt)
    raiz.addHandler(h)

logging.info("=== arranca el entrenamiento ===")
trainer.train()

23:04:10 === arranca el entrenamiento ===
23:04:10 Vocabulary size: 50002
23:04:10 Gradient checkpoint layers: []


/usr/bin/python3 /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-865e2ea1-41ad-4bfc-91ee-2b764b05b318.json


23:04:10 Gradient checkpoint layers: []
23:04:10 Set decoder to none as ctc_weight==1.0
Parámetros totales     : 64,456,018
Parámetros entrenables : 64,456,018 (100.0 %)
23:04:11 pytorch.version=2.11.0+cu128, cuda.available=True, cudnn.version=91900, cudnn.benchmark=False, cudnn.deterministic=False
23:04:11 Model structure:
ESPnetS2TModel(
  (frontend): None
  (normalize): None
  (encoder): EBranchformerEncoder(
    (embed): Conv2dSubsampling(
      (conv): Sequential(
        (0): Conv2d(1, 384, kernel_size=(3, 3), stride=(2, 2))
        (1): ReLU()
        (2): Conv2d(384, 384, kernel_size=(3, 3), stride=(2, 2))
        (3): ReLU()
      )
      (out): Sequential(
        (0): Linear(in_features=48768, out_features=384, bias=True)
        (1): PositionalEncoding(
          (dropout): Dropout(p=0.05, inplace=False)
        )
      )
    )
    (encoders): MultiSequential(
      (0): EBranchformerEncoderLayer(
        (attn): MultiHeadedAttention(
          (linear_q): Linear(in_feature

/usr/local/lib/python3.13/dist-packages/espnet2/train/trainer.py:219: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


23:04:16 1/20epoch started


/usr/local/lib/python3.13/dist-packages/espnet2/train/trainer.py:638: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(


23:04:24 1epoch:train:1-20batch: iter_time=0.005, forward_time=0.116, loss_ctc=31.002, loss=31.002, backward_time=0.105, grad_norm=2.340e+03, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.023, optim0_lr0=3.250e-05, train_time=0.743
23:04:30 1epoch:train:21-40batch: iter_time=1.653e-04, forward_time=0.088, loss_ctc=23.763, loss=23.763, backward_time=0.083, grad_norm=194.692, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.006, optim0_lr0=8.250e-05, train_time=0.572
23:04:36 1epoch:train:41-60batch: iter_time=1.914e-04, forward_time=0.096, loss_ctc=25.214, loss=25.214, backward_time=0.088, grad_norm=123.107, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.006, optim0_lr0=1.325e-04, train_time=0.620
23:04:42 1epoch:train:61-80batch: iter_time=1.660e-04, forward_time=0.095, loss_ctc=21.445, loss=21.445, backward_time=0.083, grad_norm=124.528, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.006, optim0_lr0=1.825e-04, train_time=0.587
23:04:48 1epoch:train:81-100bat

/usr/local/lib/python3.13/dist-packages/espnet2/train/trainer.py:856: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(


23:09:15 1epoch results: [train] iter_time=2.836e-04, forward_time=0.097, loss_ctc=21.870, loss=21.870, backward_time=0.088, grad_norm=143.244, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.006, optim0_lr0=6.689e-04, train_time=0.627, time=4 minutes and 38.93 seconds, total_count=888, gpu_max_cached_mem_GB=20.996, [valid] loss_ctc=49.559, cer_ctc=0.941, loss_att=nan, acc=nan, cer=nan, wer=nan, loss=49.559, time=19.99 seconds, total_count=163, gpu_max_cached_mem_GB=21.617
23:09:19 The best model has been updated: valid.loss
23:09:19 2/20epoch started. Estimated time to finish: 1 hour, 35 minutes and 50.89 seconds
23:09:26 2epoch:train:1-20batch: iter_time=0.006, forward_time=0.103, loss_ctc=17.454, loss=17.454, backward_time=0.105, grad_norm=81.002, clip=100.000, loss_scale=6.554e+04, optim_step_time=0.006, optim0_lr0=6.663e-04, train_time=0.727
23:09:33 2epoch:train:21-40batch: iter_time=1.703e-04, forward_time=0.098, loss_ctc=19.302, loss=19.302, backward_time=0.085, grad_norm

## 8. Métricas

In [12]:
filas = []
for linea in open(LOG_PATH, errors="ignore"):
    m = re.search(r"(\d+)epoch results:(.*)", linea)
    if not m:
        continue
    ep, resto = int(m.group(1)), m.group(2)
    fila = {"epoch": ep}
    for split in ("train", "valid"):
        trozo = re.search(rf"\[{split}\](.*?)(?=\[|$)", resto)
        if trozo:
            for k, v in re.findall(r"(\w+)=([-\d.]+(?:[eE][-+]?\d+)?)", trozo.group(1)):
                fila[f"{split}_{k}"] = float(v)
    filas.append(fila)

hist = pd.DataFrame(filas).drop_duplicates("epoch").sort_values("epoch")
cols = [c for c in ["epoch", "train_loss", "valid_loss", "train_loss_ctc",
                    "valid_loss_ctc", "valid_cer_ctc"] if c in hist.columns]
display(hist[cols].tail(10))

,epoch,train_loss,valid_loss,train_loss_ctc,valid_loss_ctc,valid_cer_ctc
10,11,18.795,50.430,18.795,50.430,0.932
11,12,18.731,51.056,18.731,51.056,0.924
12,13,18.625,52.144,18.625,52.144,0.904
13,14,18.554,51.349,18.554,51.349,0.929
14,15,18.479,52.192,18.479,52.192,0.912
15,16,18.392,52.807,18.392,52.807,0.922
16,17,18.320,53.033,18.320,53.033,0.880
17,18,18.272,53.597,18.272,53.597,0.939
18,19,18.208,53.667,18.208,53.667,0.928
19,20,18.128,53.617,18.128,53.617,0.902


In [13]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

for c, col in [("train_loss", "steelblue"), ("valid_loss", "indianred")]:
    if c in hist: ax[0].plot(hist["epoch"], hist[c], label=c, color=col)
ax[0].set_xlabel("época"); ax[0].set_ylabel("pérdida"); ax[0].legend()
ax[0].set_title("Pérdida"); ax[0].grid(alpha=.3)

if "valid_cer_ctc" in hist:
    ax[1].plot(hist["epoch"], hist["valid_cer_ctc"], color="darkgreen")
    mejor = hist["valid_cer_ctc"].min()
    ax[1].axhline(mejor, ls="--", c="gray")
    ax[1].text(hist["epoch"].iloc[0], mejor, f"  mínimo {mejor:.3f}", va="bottom")
ax[1].set_xlabel("época"); ax[1].set_ylabel("CER"); ax[1].set_ylim(0, 1.6)
ax[1].set_title("CER de la rama CTC (validación)"); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 9. Inferencia

Con `ctc_weight = 1.0` el decodificador autorregresivo no participa en la
pérdida, así que la salida del sistema es la de la rama CTC. Decodificamos con
la regla voraz: el token más probable en cada frame, y luego la función de
colapso de CTC.

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = next(p for p in [f"{EXP_DIR}/valid.loss.best.pth",
                        f"{EXP_DIR}/valid.acc.best.pth",
                        f"{EXP_DIR}/train.loss.best.pth"] if os.path.exists(p))
print("Checkpoint:", os.path.basename(ckpt))

modelo = build_model_fn(None)
modelo.load_state_dict(torch.load(ckpt, map_location="cpu"))
modelo.to(device).eval()

Checkpoint: valid.loss.best.pth
00:44:33 Vocabulary size: 50002
00:44:34 Gradient checkpoint layers: []
00:44:34 Gradient checkpoint layers: []
00:44:35 Set decoder to none as ctc_weight==1.0
Parámetros totales     : 64,456,018
Parámetros entrenables : 64,456,018 (100.0 %)


ESPnetS2TModel(
  (frontend): None
  (normalize): None
  (encoder): EBranchformerEncoder(
    (embed): Conv2dSubsampling(
      (conv): Sequential(
        (0): Conv2d(1, 384, kernel_size=(3, 3), stride=(2, 2))
        (1): ReLU()
        (2): Conv2d(384, 384, kernel_size=(3, 3), stride=(2, 2))
        (3): ReLU()
      )
      (out): Sequential(
        (0): Linear(in_features=48768, out_features=384, bias=True)
        (1): PositionalEncoding(
          (dropout): Dropout(p=0.05, inplace=False)
        )
      )
    )
    (encoders): MultiSequential(
      (0): EBranchformerEncoderLayer(
        (attn): MultiHeadedAttention(
          (linear_q): Linear(in_features=384, out_features=384, bias=True)
          (linear_k): Linear(in_features=384, out_features=384, bias=True)
          (linear_v): Linear(in_features=384, out_features=384, bias=True)
          (linear_out): Linear(in_features=384, out_features=384, bias=True)
          (dropout): Dropout(p=0.05, inplace=False)
          (

In [ ]:
N = min(150, len(val_trials))
filas = []
for t in val_trials[:N]:
    hyp, blanco, _ = eeg.decodificar_voraz(
        modelo, t["input_features"], device, tokenizer, converter,
        blank=modelo.blank_id)
    filas.append({
        "referencia": eeg.normalizar_para_evaluar(t["text_raw"]),
        "hipótesis":  eeg.normalizar_para_evaluar(hyp),
        "blancos":    blanco,
        "cer":        eeg.tasa_error(t["text_raw"], hyp, "caracter"),
        "wer":        eeg.tasa_error(t["text_raw"], hyp, "palabra"),
    })
pred = pd.DataFrame(filas)

print(f"Evaluado sobre {len(pred)} trials de validación\n")
print(f"  CER  : {pred['cer'].mean():.3f}")
print(f"  WER  : {pred['wer'].mean():.3f}")
print(f"  Tasa media de blancos por frame : {pred['blancos'].mean():.1%}")
print(f"  Hipótesis vacías                : {(pred['hipótesis'].str.len()==0).mean():.1%}")
print(f"  Hipótesis distintas             : {pred['hipótesis'].nunique()} de {len(pred)}")

In [16]:
pd.set_option("display.max_colwidth", 60)
display(pred[["referencia", "hipótesis", "cer", "wer"]].head(15))

,referencia,hipótesis,cer,wer
0,you can see the code at this point as well,,1.0,1.0
1,how does it keep the cost down,,1.0,1.0
2,not too controversial,,1.0,1.0
3,the jury and a judge work together on it,,1.0,1.0
4,were quite vocal about it,,1.0,1.0
5,he said the decision to part ways was mutual,,1.0,1.0
6,in fact this morning when they were talking,,1.0,1.0
7,this is like a cruelty joke,,1.0,1.0
8,has such a high clay content,,1.0,1.0
9,woodworking mastery,,1.0,1.0


## 10. Por qué esto no podía funcionar

Hasta aquí, todo lo anterior es el notebook que me pedisteis: la adaptación
hecha, el modelo entrenado y una inferencia. El resultado es malo. La pregunta
de la segunda reunión era si eso se debe a un error de programación.

No lo es, y esta sección lo demuestra en dos pasos: primero la aritmética del
problema, y después un control experimental.

### 10.1 La aritmética del vocabulario

El objetivo CTC de este notebook son las subpalabras de OWSM, que es el
vocabulario que trae el modelo. Vamos a contar cuántas hay y cuántas veces se
observa cada una.

In [17]:
V = len(converter.token_list)
todos = np.concatenate([tokenize(train_raw[i]["text_ctc"]) for i in range(len(train_raw))])
vistos, cuentas = np.unique(todos, return_counts=True)
d_model = modelo.encoder.output_size()

print("EL OBJETIVO CTC")
print("-" * 66)
print(f"  Vocabulario de OWSM                    : {V:,} clases")
print(f"  Frases de entrenamiento                : {len(train_raw):,}")
print(f"  Tokens objetivo en total               : {len(todos):,}")
print(f"  Media de tokens por frase              : {len(todos)/len(train_raw):.1f}")
print()
print(f"  Clases observadas al menos una vez     : {len(vistos):,} "
      f"({len(vistos)/V:.2%} del vocabulario)")
print(f"  Clases NUNCA observadas                : {V-len(vistos):,} "
      f"({1-len(vistos)/V:.2%})")
print(f"  Clases observadas una sola vez         : {(cuentas==1).sum():,}")
print()
print(f"  Parámetros de la cabeza CTC            : {d_model} x {V} = {d_model*V:,}")
print(f"  Observaciones por parámetro de la cabeza: {len(todos)/(d_model*V):.4f}")

EL OBJETIVO CTC
------------------------------------------------------------------
  Vocabulario de OWSM                    : 50,002 clases
  Frases de entrenamiento                : 8,072
  Tokens objetivo en total               : 53,619
  Media de tokens por frase              : 6.6

  Clases observadas al menos una vez     : 3,643 (7.29% del vocabulario)
  Clases NUNCA observadas                : 46,359 (92.71%)
  Clases observadas una sola vez         : 1,536

  Parámetros de la cabeza CTC            : 384 x 50002 = 19,200,768
  Observaciones por parámetro de la cabeza: 0.0028


In [18]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))

ax[0].bar(["observadas", "nunca vistas"], [len(vistos), V - len(vistos)],
          color=["steelblue", "lightgray"], edgecolor="black")
ax[0].set_ylabel("nº de clases"); ax[0].set_title(f"Cobertura del vocabulario (V = {V:,})")
for i, v in enumerate([len(vistos), V - len(vistos)]):
    ax[0].text(i, v, f"{v:,}", ha="center", va="bottom")

ax[1].hist(cuentas, bins=np.logspace(0, np.log10(cuentas.max()), 40),
           color="steelblue", edgecolor="white")
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_xlabel("veces que aparece la clase"); ax[1].set_ylabel("nº de clases")
ax[1].set_title("Frecuencia de las clases observadas")

plt.tight_layout(); plt.show()